# LLM Zoomcamp 2026
## Module 4 - Evaluation Homework

Author: Debabrata Mishra

This notebook demonstrates how to evaluate different retrieval methods for a Retrieval-Augmented Generation (RAG) system.

The notebook covers:

- Loading course lessons
- Chunking documents
- Generating evaluation questions
- Building keyword search
- Building vector search
- Hybrid search with Reciprocal Rank Fusion (RRF)
- Measuring retrieval performance using:
    - Hit Rate
    - Mean Reciprocal Rank (MRR)

In [1]:
import sys
import openai

print(sys.executable)
print(openai.__version__)

C:\Users\dmish\llm-zoomcamp-code\.venv\Scripts\python.exe
2.41.0


# Imports

In [2]:
import os
import json
import pandas as pd

from dotenv import load_dotenv

from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

# Load Environment Variables

In [3]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Download Helper Files

In [5]:
import evaluation_utils

print(dir(evaluation_utils))

['RAGBase', 'RAGWithUsage', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calc_price', 'calc_total_price', 'llm_structured', 'llm_structured_retry', 'map_progress', 'time', 'tqdm']


In [6]:
from pydantic import BaseModel

from evaluation_utils import llm_structured

In [7]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [8]:
from pydantic import BaseModel
from evaluation_utils import llm_structured

class Questions(BaseModel):
    questions: list[str]

print("Everything imported successfully!")

Everything imported successfully!


In [9]:
import inspect
from evaluation_utils import llm_structured

print(inspect.signature(llm_structured))

(client, instructions, user_prompt, output_type, model='gpt-5.4-mini')


In [16]:
import json

def generate_questions(document):

    user_prompt = json.dumps(
        {
            "filename": document["filename"],
            "content": document["content"],
        },
        indent=2,
    )

    response, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
    )

    return response.questions, usage

In [12]:
print(client)

In [13]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [15]:
print(len(documents))
print(documents[0])
print(documents[0].keys())

72
{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a s

In [17]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [19]:
ground_truth_small = []

token_counts = []

for document in documents[:3]:

    questions, usage = generate_questions(document)

    token_counts.append(usage.input_tokens)

    for question in questions:
        ground_truth_small.append(
            {
                "question": question,
                "filename": document["filename"]
            }
        )

In [20]:
ground_truth_small[:5]

[{'question': 'What is Retrieval-Augmented Generation, and why would you use it instead of relying on the model alone?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does the course start by building the RAG system in plain Python instead of using a framework right away?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of LLMs that RAG is meant to fix?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example project will this module build, and what will the FAQ bot answer from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'How is the module split up, and what changes in the second part?',
  'filename': '01-agentic-rag/lessons/01-intro.md'}]

In [21]:
len(ground_truth_small)


15

In [22]:
avg_input_tokens = sum(token_counts) / len(token_counts)

print(avg_input_tokens)

1357.0


# Q1 – Generate Ground Truth Questions

Before evaluating the retrieval system, we first need a **ground truth dataset** that links natural language questions to the lesson pages that answer them.

The course uses a Large Language Model (LLM) to automatically generate realistic questions that a student might ask after reading each lesson. These generated questions are then paired with the corresponding lesson filename, creating a labelled dataset for retrieval evaluation.

## Methodology

For this exercise:

1. Load the first **three lesson pages** from the LLM Zoomcamp repository.
2. Prompt the LLM to generate **five student-style questions** for each lesson.
3. Use **structured output** with a Pydantic model to ensure the response is returned as a list of questions.
4. Record the API usage statistics to measure the number of **input tokens** consumed during question generation.
5. Calculate the **average input tokens** across the three API calls.

## Why this evaluation matters

Generating high-quality ground truth questions enables us to objectively evaluate different retrieval strategies later in the homework.

Each generated question is associated with the lesson from which it was created, allowing us to determine whether a search algorithm successfully retrieves the correct document.

Although only the first three lessons are processed here (15 questions in total), the homework subsequently provides a pre-generated dataset containing **360 questions** covering all **72 lesson pages**.

## Results

- Lessons processed: **3**
- Questions generated per lesson: **5**
- Total generated questions: **15**
- Average input tokens: **1357**

Therefore, the correct answer for **Question 1** is:

> **1400** (closest multiple-choice option)

# Next Question (Q2)

## Load the Ground Truth Dataset

Instead of generating questions for all 72 lessons, we use the pre-generated dataset supplied with the homework. This contains 360 questions mapped to their corresponding lesson pages.

In [24]:
ground_truth = pd.read_csv("ground-truth.csv").to_dict(orient="records")
len(ground_truth)

360

In [25]:
print(ground_truth[:5])

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}, {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?', 'filename': '01-agentic-rag/lessons/01-intro.md'}, {'question': 'What are the main weaknesses of large language models that this module is trying to work around?', 'filename': '01-agentic-rag/lessons/01-intro.md'}, {'question': 'What will the course build in the first part of the module, and how is the second part different?', 'filename': '01-agentic-rag/lessons/01-intro.md'}, {'question': 'What kind of example app are you building here, and what data will it answer questions from?', 'filename': '01-agentic-rag/lessons/01-intro.md'}]


## Create Document Chunks

The search index operates over document chunks rather than entire lesson pages. We use the same chunking strategy from Homework 2, splitting each lesson into overlapping chunks.

In [26]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [27]:
print(len(chunks))
print(chunks[0])

295
{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour p

## Q2 – Keyword Search

Reuse the keyword search index from Homework 2 and execute a search using the first question in the ground truth dataset. The filename of the top-ranked result is the answer to Question 2.

# Build the text index

In [28]:
import minsearch

text_index = minsearch.Index(
    text_fields=["content"]
)

text_index.fit(chunks)

# Define a reusable search function

In [29]:
def text_search(query, num_results=5):
    return text_index.search(
        query,
        num_results=num_results
    )

# Get the first ground truth question

In [30]:
ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

# Extract the question

In [31]:
query = ground_truth[0]["question"]

print(query)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


# Run the search

In [32]:
results = text_search(query)

results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

# Inspect the filenames

In [33]:
for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc['filename']}")

1. 01-agentic-rag/lessons/03-rag.md
2. 01-agentic-rag/lessons/13-function-calling.md
3. 01-agentic-rag/lessons/03-rag.md
4. 01-agentic-rag/lessons/13-function-calling.md
5. 01-agentic-rag/lessons/01-intro.md


In [34]:
results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

## Q2 – Evaluate Keyword (Text) Search

In this section, we evaluate the baseline **keyword-based retrieval** system built in Homework 2.

The objective is to determine whether a traditional lexical search engine can retrieve the correct lesson page when given a natural language question from the ground truth dataset.

### Methodology

1. Select the **first question** from the `ground_truth.csv` dataset.
2. Execute the `text_search()` function.
3. Retrieve the **top 5 ranked documents**.
4. Inspect the filename of the **highest-ranked result**.
5. Compare the returned filename with the expected source lesson.

### Why this evaluation matters

Keyword search ranks documents according to **shared words and phrases** between the user's query and the indexed lesson chunks.

Unlike semantic vector search, keyword search:

- relies on exact or closely matching terms,
- cannot understand paraphrases or meaning,
- often prefers documents that repeat important keywords more frequently.

Consequently, the top-ranked document may not always be the original lesson from which the question was generated.

### Homework Question

> After running `text_search()` for the first ground truth question, what is the filename of the first returned result?

### Result

The top-ranked document returned by the keyword search is:

```text
01-agentic-rag/lessons/03-rag.md
```

This corresponds to the correct multiple-choice answer for **Question 2**.

# Next Question (Q3)

## Q3 – Evaluate Vector Search

In this section, we evaluate the **semantic vector search** system developed in Homework 2.

Unlike keyword search, vector search retrieves documents based on **semantic similarity** rather than exact word matching. Both the user's question and every document chunk are converted into dense vector embeddings. The search engine then identifies the chunks whose embeddings are closest to the query embedding in the semantic space.

### Methodology

1. Select the **same first question** from the ground truth dataset used in Question 2.
2. Generate an embedding for the question using the **all-MiniLM-L6-v2** embedding model.
3. Search the vector index for the **Top-5 most similar document chunks**.
4. Inspect the filename of the highest-ranked result.
5. Compare the retrieved lesson with the expected source document.

### Why this evaluation matters

Unlike traditional keyword search, semantic vector search can recognise documents that express the same concept using different words or phrasing. This enables the retrieval system to:

- understand paraphrases,
- capture semantic meaning,
- retrieve conceptually related documents,
- improve retrieval quality for natural language questions.

The top-ranked document should therefore be semantically related to the user's question, even if it does not share many exact keywords.

### Homework Question

> After running `vector_search()` for the first ground truth question, what is the filename of the first returned result?

In [ ]:
## Load the Embedding Model

The vector search implementation requires the same embedding model used in Homework 2.

We load the locally downloaded ONNX version of the **all-MiniLM-L6-v2** model and create a reusable `Embedder` instance. This model is used to generate embeddings for user queries before searching the vector index.

# Create the Embedder

In [38]:
import os
import sys

# Add project root to Python path
PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from embedder import Embedder

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "Xenova",
    "all-MiniLM-L6-v2"
)

embedder = Embedder(path=MODEL_PATH)

print("Embedder loaded successfully!")

Embedder loaded successfully!


# Build the Vector Embeddings

In [39]:
chunk_texts = [
    chunk["content"]
    for chunk in chunks
]

X = embedder.encode_batch(chunk_texts)

print(X.shape)

(295, 384)


# Build the Vector Index

In [40]:
import minsearch

vector_index = minsearch.VectorSearch()

vector_index.fit(
    vectors=X,
    payload=chunks
)

# Define the Search Function

In [41]:
def vector_search(query, num_results=5):

    query_vector = embedder.encode(query)

    return vector_index.search(
        query_vector,
        num_results=num_results
    )

In [44]:
query = ground_truth[0]["question"]

results = vector_search(query)

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc['filename']}")

1. 01-agentic-rag/lessons/01-intro.md
2. 04-evaluation/lessons/11-evaluation-intro.md
3. 04-evaluation/lessons/12-rag-answers.md
4. 01-agentic-rag/lessons/10-rag-next-steps.md
5. 06-best-practices/lessons/01-intro.md


In [45]:
results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

## Q3 – Evaluate Semantic Vector Search

In this section, we evaluate the **semantic vector search** system developed in Homework 2.

Unlike keyword search, vector search retrieves documents based on **semantic meaning** rather than exact word matching. Both the user's question and every document chunk are converted into dense vector embeddings using the **all-MiniLM-L6-v2** embedding model. The search engine then retrieves the chunks whose embeddings are most similar to the query embedding using cosine similarity.

### Methodology

1. Reuse the **same first question** from the ground truth dataset used in Question 2.
2. Generate a vector embedding for the question using the ONNX embedding model.
3. Search the MinSearch **VectorSearch** index for the Top-5 most semantically similar document chunks.
4. Inspect the filename of the highest-ranked result.
5. Compare the retrieved lesson with the original lesson from which the question was generated.

### Why this evaluation matters

Unlike keyword search, semantic vector search understands the **meaning** of the query instead of relying solely on matching words.

This enables the retrieval system to:

- recognise paraphrases,
- retrieve conceptually related information,
- handle different wording of the same idea,
- improve retrieval quality for natural language questions.

For Retrieval-Augmented Generation (RAG) systems, semantic search is often more effective than keyword search because users rarely phrase their questions using exactly the same words as the source documents.

### Homework Question

> After running `vector_search()` for the first ground truth question, what is the filename of the first returned result?

### Results

Top-5 retrieved lesson chunks:

1. `01-agentic-rag/lessons/01-intro.md`
2. `04-evaluation/lessons/11-evaluation-intro.md`
3. `04-evaluation/lessons/12-rag-answers.md`
4. `01-agentic-rag/lessons/10-rag-next-steps.md`
5. `06-best-practices/lessons/01-intro.md`

The highest-ranked result is:

```text
01-agentic-rag/lessons/01-intro.md
```

This matches the lesson from which the ground truth question was originally generated, demonstrating that semantic vector search successfully retrieved the correct source document.

Therefore, the correct answer for **Question 3** is:

> **01-agentic-rag/lessons/01-intro.md**

# Interpretation (optional)

Notice the contrast between Q2 and Q3:

Search Method	Top Result
Keyword Search	01-agentic-rag/lessons/03-rag.md
Vector Search	01-agentic-rag/lessons/01-intro.md

This nicely illustrates one of the main ideas of the course:

Keyword search favoured 03-rag.md because it contains the phrase "retrieval-augmented generation" more frequently.
Vector search correctly identified 01-intro.md as the best semantic match because the entire question is about the concepts introduced in that lesson, not just the presence of specific keywords.

This comparison provides a strong motivation for the later homework questions, where you'll quantify retrieval quality using Hit Rate, Mean Reciprocal Rank (MRR), and finally Hybrid Search (RRF).

# Next Question (Q4)

## Q4 – Evaluate Keyword Search using Hit Rate

In this section, we evaluate the overall performance of the **keyword-based retrieval system** using the complete ground truth dataset.

Rather than examining a single query, we measure how frequently the search engine retrieves the correct lesson page across **all 360 evaluation questions**.

### Evaluation Metric – Hit Rate

**Hit Rate (HR)** measures the proportion of queries for which the correct document appears anywhere within the retrieved Top-*k* results.

For this homework, each query returns the **Top-5** retrieved document chunks.

The Hit Rate is calculated as:

\[
\text{Hit Rate} =
\frac{\text{Number of successful retrievals}}
{\text{Total number of queries}}
\]

where a **successful retrieval** means that the expected lesson filename appears within the Top-5 retrieved results.

### Methodology

1. Iterate over every question in the ground truth dataset.
2. Execute `text_search()` for each question.
3. Retrieve the Top-5 ranked lesson chunks.
4. Compare the retrieved filenames with the expected filename.
5. Compute the overall Hit Rate.

### Why this metric matters

Hit Rate measures the **recall capability** of a retrieval system.

A higher Hit Rate indicates that the correct document is usually retrieved somewhere within the candidate set, even if it is not ranked first. This is an important property for Retrieval-Augmented Generation (RAG) systems, since the LLM can still generate a correct answer if the relevant context is included among the retrieved documents.

The next question (Q5) introduces **Mean Reciprocal Rank (MRR)**, which additionally considers **where** the correct document appears in the ranking.

# Evaluate all questions

In [46]:
text_search_results = []

for record in ground_truth:

    question = record["question"]
    filename = record["filename"]

    results = text_search(question)

    retrieved = [
        doc["filename"]
        for doc in results
    ]

    text_search_results.append(
        {
            "question": question,
            "expected": filename,
            "retrieved": retrieved,
        }
    )

# Compute Hit Rate

# Since I have downloaded evaluation_utils.py, use the helper function it provides instead of writing the metric manually.

# First, inspect what's available:

In [47]:
dir(evaluation_utils)

['RAGBase',
 'RAGWithUsage',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'calc_price',
 'calc_total_price',
 'llm_structured',
 'llm_structured_retry',
 'map_progress',
 'time',
 'tqdm']

In [50]:
from evaluation_utils import *



In [51]:
def hit_rate(results):

    successes = 0

    for item in results:
        if item["expected"] in item["retrieved"]:
            successes += 1

    return successes / len(results)

In [52]:
hr = hit_rate(text_search_results)

print(hr)

0.7583333333333333


## Q4 – Evaluate Keyword Search using Hit Rate

In this section, we evaluate the overall performance of the **keyword-based retrieval system** using the complete ground truth dataset.

Rather than analysing a single query, we assess how effectively the search engine retrieves the correct lesson page across all **360 evaluation questions**.

### Evaluation Metric – Hit Rate

Hit Rate measures the proportion of queries for which the expected document appears within the retrieved **Top-5** search results.

For each question:

1. Execute the `text_search()` function.
2. Retrieve the Top-5 ranked lesson chunks.
3. Compare the expected lesson filename with the retrieved filenames.
4. Count the query as a **hit** if the expected filename appears anywhere within the Top-5 results.

The Hit Rate is calculated as:

\[
\text{Hit Rate} =
\frac{\text{Number of successful retrievals}}
{\text{Total number of evaluation queries}}
\]

### Why this metric matters

Hit Rate evaluates the retrieval system's ability to include the correct document within its candidate set.

Unlike accuracy at rank one, Hit Rate does not require the correct document to be ranked first—it only needs to appear somewhere among the retrieved results.

For Retrieval-Augmented Generation (RAG) systems, this is an important property because the language model can still generate a correct answer if the relevant document is included in the retrieved context.

### Results

- Evaluation dataset: **360 questions**
- Retrieval method: **Keyword (Text) Search**
- Retrieved documents per query: **Top-5**

Computed Hit Rate:

```text
0.7583333333333333
```

Rounded to two decimal places:

```text
0.76
```

Therefore, the correct answer for **Question 4** is:

> **0.76**

# Interpretation

# Notice the progression so far:

Question	Metric	Result
Q2	Top-1 Keyword Search	03-rag.md
Q3	Top-1 Vector Search	01-intro.md
Q4	Keyword Search Hit Rate	0.7583 (≈ 0.76)

This tells us that although the keyword search did not rank the correct lesson first for the initial example (Q2), it still retrieves the correct lesson within the Top-5 results for about 76% of all evaluation questions. The next question (Q5) will use Mean Reciprocal Rank (MRR), which is a stricter metric because it rewards systems that rank the correct document higher, not just somewhere in the result list.

# Next Question (Q5)

## Q5 – Evaluate Semantic Vector Search using Mean Reciprocal Rank (MRR)

In this section, we evaluate the ranking quality of the **semantic vector search** system using the complete ground truth dataset.

Unlike Hit Rate, which only checks whether the correct document appears somewhere within the retrieved results, **Mean Reciprocal Rank (MRR)** measures **how highly the correct document is ranked**.

### Evaluation Metric – Mean Reciprocal Rank (MRR)

For each evaluation query:

1. Execute the `vector_search()` function.
2. Retrieve the **Top-5** ranked lesson chunks.
3. Find the position of the expected lesson filename.
4. Compute the reciprocal of its rank.

The reciprocal rank for an individual query is:

- Rank 1 → **1.0**
- Rank 2 → **0.5**
- Rank 3 → **0.333**
- Rank 4 → **0.25**
- Rank 5 → **0.20**
- Not retrieved → **0**

The overall MRR is calculated as the average reciprocal rank across all evaluation queries.

### Why this metric matters

MRR is a stricter evaluation metric than Hit Rate because it rewards retrieval systems that rank the correct document **as close to the top of the search results as possible**.

A system that consistently returns the correct document in the first position achieves a much higher MRR than one that returns it in lower positions, even if both systems have similar Hit Rates.

Since Retrieval-Augmented Generation (RAG) typically uses only the highest-ranked retrieved documents to construct the LLM prompt, MRR provides a better indication of practical retrieval quality.

# Evaluate Vector Search

In [53]:
vector_search_results = []

for record in ground_truth:

    question = record["question"]
    filename = record["filename"]

    results = vector_search(question)

    retrieved = [
        doc["filename"]
        for doc in results
    ]

    vector_search_results.append(
        {
            "question": question,
            "expected": filename,
            "retrieved": retrieved,
        }
    )

# Compute MRR

In [55]:
def mrr(results):

    total = 0

    for item in results:

        expected = item["expected"]
        retrieved = item["retrieved"]

        score = 0

        for rank, filename in enumerate(retrieved, start=1):
            if filename == expected:
                score = 1 / rank
                break

        total += score

    return total / len(results)

In [56]:
mrr_score = mrr(vector_search_results)

print(mrr_score)

0.5486111111111112


## Q5 – Evaluate Semantic Vector Search using Mean Reciprocal Rank (MRR)

In this section, we evaluate the ranking performance of the **semantic vector search** system using the complete ground truth dataset.

Unlike Hit Rate, which only measures whether the correct document appears anywhere within the retrieved results, **Mean Reciprocal Rank (MRR)** evaluates **how highly the correct document is ranked**. It rewards retrieval systems that consistently place the correct document near the top of the search results.

### Evaluation Metric – Mean Reciprocal Rank (MRR)

For each question in the evaluation dataset:

1. Execute the `vector_search()` function.
2. Retrieve the **Top-5** ranked lesson chunks.
3. Determine the rank position of the expected lesson filename.
4. Compute the reciprocal of that rank.
5. Average the reciprocal ranks across all evaluation questions.

The reciprocal rank for an individual query is:

| Rank | Reciprocal Rank |
|------:|----------------:|
| 1 | 1.000 |
| 2 | 0.500 |
| 3 | 0.333 |
| 4 | 0.250 |
| 5 | 0.200 |
| Not Retrieved | 0.000 |

The overall Mean Reciprocal Rank (MRR) is the average of these reciprocal ranks over the entire evaluation dataset.

### Why this metric matters

MRR is a stricter evaluation metric than Hit Rate because it considers the **position** of the correct result.

A retrieval system that consistently ranks the correct lesson first will achieve a much higher MRR than one that returns the same lesson lower in the ranking.

Since Retrieval-Augmented Generation (RAG) systems typically include only the highest-ranked retrieved documents in the LLM prompt, MRR is a strong indicator of practical retrieval effectiveness.

### Results

- Evaluation dataset: **360 questions**
- Retrieval method: **Semantic Vector Search**
- Retrieved documents per query: **Top-5**

Computed Mean Reciprocal Rank:

```text
0.5486111111111112
```

Rounded to two decimal places:

```text
0.55
```

Therefore, the correct answer for **Question 5** is:

> **0.55**

# Interpretation

# My results so far are:

Question	Retrieval Method	Evaluation Metric	Result
Q2	Keyword Search	Top-1 Result	03-rag.md
Q3	Vector Search	Top-1 Result	01-intro.md
Q4	Keyword Search	Hit Rate	0.7583 (≈ 0.76)
Q5	Vector Search	MRR	0.5486 (≈ 0.55)

These metrics show different aspects of retrieval quality:

Hit Rate (Q4) measures whether the correct document is included in the Top-5 results.
MRR (Q5) measures how highly the correct document is ranked within those results.

This distinction is important in RAG systems because ranking the correct document first generally provides the LLM with more relevant context.

# Next Question (Q6)

## Q6 – Evaluate Hybrid Search using Reciprocal Rank Fusion (RRF)

In this final experiment, we evaluate a **hybrid retrieval system** that combines the results of keyword search and semantic vector search using **Reciprocal Rank Fusion (RRF)**.

Hybrid search aims to leverage the strengths of both retrieval methods:

- **Keyword Search** captures exact lexical matches.
- **Vector Search** captures semantic similarity.

Rather than choosing one retrieval method, RRF merges the ranked lists produced by both systems into a single ranking.

### Reciprocal Rank Fusion (RRF)

For each retrieved document, RRF computes a score:

\[
\text{RRF Score}
=
\sum_i
\frac{1}{k + rank_i}
\]

where

- **rankᵢ** is the document's rank in retrieval method *i*
- **k** is a smoothing constant

Documents with larger combined scores are ranked higher.

### Why vary k?

The parameter **k** controls how much influence is given to highly-ranked documents.

- Small values of **k** place much greater emphasis on documents appearing near the top of each ranking.
- Larger values reduce the importance of rank differences and produce a more balanced combination.

The objective is to determine which **k** value produces the highest Mean Reciprocal Rank (MRR) on the evaluation dataset.

The homework compares the following values:

- 1
- 50
- 100
- 200

# Add the RRF function

In [65]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

# Modify the Hybrid Search

In [66]:
def hybrid_search(query, k=60, num_results=5):

    text_results = text_search(query, num_results)

    vector_results = vector_search(query, num_results)

    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=num_results
    )

# Evaluation function

In [67]:
def evaluate_hybrid(k):

    evaluation = []

    for row in ground_truth:

        results = hybrid_search(
            row["question"],
            k=k
        )

        retrieved = [
            doc["filename"]
            for doc in results
        ]

        evaluation.append(
            {
                "question": row["question"],
                "expected": row["filename"],
                "retrieved": retrieved,
            }
        )

    return mrr(evaluation)

# Evaluate every k

In [69]:
ks = [1, 50, 100, 200]

for k in ks:
    score = evaluate_hybrid(k)
    print(f"k={k:<3}  MRR={score:.6f}")

k=1    MRR=0.645833
k=50   MRR=0.646759
k=100  MRR=0.646759
k=200  MRR=0.646759


## Q6 – Evaluate Hybrid Search using Reciprocal Rank Fusion (RRF)

In this final experiment, we evaluate a **hybrid retrieval system** that combines the results of keyword search and semantic vector search using **Reciprocal Rank Fusion (RRF)**.

Hybrid retrieval leverages the complementary strengths of both search methods:

- **Keyword Search** retrieves documents based on lexical similarity.
- **Semantic Vector Search** retrieves documents based on semantic meaning.

Instead of choosing one ranking over the other, RRF combines both ranked lists into a single fused ranking.

### Reciprocal Rank Fusion (RRF)

For each retrieved document, the RRF score is computed as:

\[
\text{RRF Score} =
\sum_i
\frac{1}{k + rank_i}
\]

where:

- \(rank_i\) is the document's rank in retrieval system *i*,
- \(k\) is a smoothing constant that controls how strongly higher-ranked documents are favoured.

Documents are then sorted by their combined RRF score.

### Experiment

The hybrid search system was evaluated using four different values of the RRF parameter:

- k = 1
- k = 50
- k = 100
- k = 200

For each value of **k**, the Mean Reciprocal Rank (MRR) was computed over the complete evaluation dataset of **360 questions**.

### Results

| RRF k | MRR |
|------:|----:|
| 1 | 0.645833 |
| 50 | **0.646759** |
| 100 | **0.646759** |
| 200 | **0.646759** |

The highest MRR is achieved for **k = 50**.

Although **k = 100** and **k = 200** produce the same MRR, **50** is the first parameter value that reaches the maximum score and is therefore the correct homework answer.

### Conclusion

**Answer to Question 6:** **50**

# Why are 50, 100 and 200 identical?

This is due to the nature of Reciprocal Rank Fusion (RRF) and your experiment:

Each search returns only Top-5 results.
The document ranks are therefore only between 0–4 (or effectively 1–5, depending on implementation).
When k is much larger than the maximum rank (e.g. 50, 100, 200), the score differences become very small:

For example:

Rank	k=50	k=100	k=200
1	1/51	1/101	1/201
2	1/52	1/102	1/202
3	1/53	1/103	1/203

Although the absolute values differ, the relative ordering of documents often remains unchanged, so the final ranking—and therefore the MRR—stays the same.

With k = 1, the reciprocal scores vary much more between adjacent ranks, so the fused ranking changes slightly, giving a slightly lower MRR in your experiment.

# Final Homework Answers

Question	Answer
Q1	1400
Q2	01-agentic-rag/lessons/03-rag.md
Q3	01-agentic-rag/lessons/01-intro.md
Q4	0.76
Q5	0.55
Q6	50